# Phase 3.5 : Export TFLite et exploration ouverte

Chats vs chiens (Microsoft Cats vs Dogs Dataset). Suite de `phase3_4_comparaison_cross_modeles.ipynb`.

**Ce notebook a besoin de `model_tl.keras`** (sauvegardé en phase 3.3). La section "Reprise" ci-dessous refait le setup des phases 1.1 et 1.2 (nécessaire pour reconstruire `val_ds_tl`, utilisé comme jeu de calibration pour la quantization INT8 et pour tester l'inférence).

## Reprise (setup phases 1.1 et 1.2 (pipeline MobileNetV2), condensé)

Nécessite un `kaggle.json` (voir phase 1.1 pour l'obtenir).

In [ ]:
CLASS_A = "cat"
CLASS_B = "dog"
DATA_ROOT = "/content/data"

In [ ]:
!pip install kaggle -q
!mkdir -p ~/.kaggle

In [ ]:
from google.colab import files
files.upload()  # sélectionner kaggle.json

import os
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
os.rename("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

In [ ]:
!kaggle datasets download -d shaunthesheep/microsoft-catsvsdogs-dataset -p /content/data/
!cd /content/data && unzip -q microsoft-catsvsdogs-dataset.zip

In [ ]:
import os, shutil, random
import tensorflow as tf

RAW_DIR = os.path.join(DATA_ROOT, "PetImages")
raw_dirs = {CLASS_A: os.path.join(RAW_DIR, "Cat"), CLASS_B: os.path.join(RAW_DIR, "Dog")}


def list_valid_images(folder):
    # Ni PIL.Image.verify() ni la verification d'entete JFIF ne suffisent : ce dataset
    # contient quelques fichiers que seul le decodeur TensorFlow rejette au moment de
    # l'entrainement (InvalidArgumentError "Unknown image file format"). On teste donc
    # directement avec tf.io.decode_image, le meme decodeur utilise par
    # image_dataset_from_directory en phase 1.2.
    valid = []
    for fname in os.listdir(folder):
        path = os.path.join(folder, fname)
        if os.path.getsize(path) == 0:
            continue
        try:
            img_bytes = tf.io.read_file(path)
            tf.io.decode_image(img_bytes, channels=3)
        except Exception:
            continue
        valid.append(path)
    return valid


files_a = list_valid_images(raw_dirs[CLASS_A])
files_b = list_valid_images(raw_dirs[CLASS_B])

for split in ["train", "val"]:
    for cls in [CLASS_A, CLASS_B]:
        os.makedirs(os.path.join(DATA_ROOT, split, cls), exist_ok=True)

random.seed(42)
random.shuffle(files_a)
random.shuffle(files_b)


def split_and_copy(file_list, cls):
    cut = int(len(file_list) * 0.8)
    train_files, val_files = file_list[:cut], file_list[cut:]
    for f in train_files:
        shutil.copy(f, os.path.join(DATA_ROOT, "train", cls, os.path.basename(f)))
    for f in val_files:
        shutil.copy(f, os.path.join(DATA_ROOT, "val", cls, os.path.basename(f)))


split_and_copy(files_a, CLASS_A)
split_and_copy(files_b, CLASS_B)
print(f"{CLASS_A}: {len(files_a)} images valides")
print(f"{CLASS_B}: {len(files_b)} images valides")

In [ ]:
IMG_SIZE_TL = (160, 160)
BATCH_SIZE = 32

train_ds_tl = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_ROOT, "train"),
    labels="inferred",
    label_mode="binary",
    class_names=[CLASS_A, CLASS_B],
    color_mode="rgb",
    image_size=IMG_SIZE_TL,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42,
)
train_ds_tl = train_ds_tl.apply(tf.data.experimental.ignore_errors())

val_ds_tl = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_ROOT, "val"),
    labels="inferred",
    label_mode="binary",
    class_names=[CLASS_A, CLASS_B],
    color_mode="rgb",
    image_size=IMG_SIZE_TL,
    batch_size=BATCH_SIZE,
    shuffle=False,
    seed=42,
)
val_ds_tl = val_ds_tl.apply(tf.data.experimental.ignore_errors())

# preprocess_input de MobileNetV2 normalise dans [-1, 1] (pas [0, 1]) : pas de
# Rescaling(1./255) ici, preprocess_input le remplace.
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input
train_ds_tl = train_ds_tl.map(lambda x, y: (preprocess_input(x), y))
val_ds_tl = val_ds_tl.map(lambda x, y: (preprocess_input(x), y))

AUTOTUNE = tf.data.AUTOTUNE
train_ds_tl = train_ds_tl.cache().prefetch(buffer_size=AUTOTUNE)
val_ds_tl = val_ds_tl.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
import os
import tensorflow as tf

if not os.path.exists("model_tl.keras"):
    print("model_tl.keras est manquant.")
    print("Execute phase3_3_finetuning.ipynb dans ce runtime avant de continuer,")
    print("ou uploade le fichier manuellement :")
    print('  from google.colab import files; files.upload()')
else:
    model_tl = tf.keras.models.load_model("model_tl.keras")
    print("model_tl.keras charge.")

## Phase 3.5 : Export TFLite et exploration ouverte

**Objectif** : convertir le modèle fine-tuné en format TFLite, mesurer la compression, puis explorer librement.

Il n'y a pas d'état "terminé" ici. Cette phase n'a pas de plafond.

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model_tl)
tflite_model = converter.convert()

tflite_path = f"/content/{CLASS_A}_vs_{CLASS_B}_mobilenet.tflite"
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

size_tflite = os.path.getsize(tflite_path) / (1024 * 1024)
size_keras = os.path.getsize("model_tl.keras") / (1024 * 1024)
print(f"Taille Keras : {size_keras:.1f} Mo")
print(f"Taille TFLite : {size_tflite:.1f} Mo")
print(f"Facteur de compression : {size_keras / size_tflite:.1f}x")

### Inférence avec le modèle TFLite

On reconstruit rapidement `val_ds_tl` (Reprise) pour prendre une image de test.

In [ ]:
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

images, labels = next(iter(val_ds_tl))
test_image = images[0:1].numpy()
true_label = int(labels[0].numpy()[0])

interpreter.set_tensor(input_details[0]["index"], test_image)
interpreter.invoke()
prediction = interpreter.get_tensor(output_details[0]["index"])[0][0]

print(f"Prediction : {prediction:.3f} (0={CLASS_A}, 1={CLASS_B})")
print(f"Vraie classe : {CLASS_A if true_label == 0 else CLASS_B}")
print(f"Confiance : {max(prediction, 1 - prediction):.1%}")

### Directions d'exploration (choisis ce qui t'intéresse)

**Direction A : quantization INT8** — ci-dessous, prête à exécuter.

**Direction B : architecture alternative** — remplace MobileNetV2 par EfficientNetB0, VGG16 ou ResNet50 (`weights="imagenet"`, `include_top=False`, adapter `input_shape` à 224x224 pour VGG16/ResNet50), reprends les phases 3.1 à 3.3, ajoute une ligne au tableau de comparaison.

**Direction C : push de la précision** — dégèle plus de couches, réduis encore le learning rate, augmente l'input à 224x224, ajoute `ReduceLROnPlateau`, ajuste le seuil de Dropout.

**Direction D : autre dataset** — teste le même pipeline sur un second sujet de classification binaire, visuellement différent.

In [ ]:
converter_int8 = tf.lite.TFLiteConverter.from_keras_model(model_tl)
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]


def representative_dataset():
    for images, _ in val_ds_tl.take(50):
        yield [images.numpy()]


converter_int8.representative_dataset = representative_dataset
tflite_int8 = converter_int8.convert()

tflite_int8_path = f"/content/{CLASS_A}_vs_{CLASS_B}_mobilenet_int8.tflite"
with open(tflite_int8_path, "wb") as f:
    f.write(tflite_int8)

size_int8 = os.path.getsize(tflite_int8_path) / (1024 * 1024)
print(f"Taille TFLite FP32 : {size_tflite:.1f} Mo")
print(f"Taille TFLite INT8 : {size_int8:.1f} Mo")
print(f"Facteur de compression supplementaire : {size_tflite / size_int8:.1f}x")

In [ ]:
# Mesure de l'accuracy_drop : evalue le modele TFLite INT8 sur quelques batches du val set
# et compare avec model_tl.evaluate() pour decider si le tradeoff (taille vs precision)
# vaut le coup sur ce dataset.
interpreter_int8 = tf.lite.Interpreter(model_path=tflite_int8_path)
interpreter_int8.allocate_tensors()
input_details_int8 = interpreter_int8.get_input_details()
output_details_int8 = interpreter_int8.get_output_details()

correct = 0
total = 0
for images, labels in val_ds_tl.take(20):
    for img, label in zip(images.numpy(), labels.numpy()):
        interpreter_int8.set_tensor(input_details_int8[0]["index"], img[None, ...])
        interpreter_int8.invoke()
        pred = interpreter_int8.get_tensor(output_details_int8[0]["index"])[0][0]
        correct += int((pred >= 0.5) == bool(label[0]))
        total += 1

print(f"Accuracy TFLite INT8 sur {total} images de val : {correct / total:.1%}")

### Vérifications avant de conclure

- **Happy path** : fichier `.tflite` produit avec le bon nom, taille affichée, facteur de compression calculé, une inférence TFLite exécutée avec succès, direction A (quantization) entamée.
- **Edge case** : la quantization INT8 divise la taille par ~4 mais peut coûter 1-2 points de `val_accuracy` — la cellule d'évaluation ci-dessus permet de mesurer ce tradeoff précisément sur ce dataset plutôt que de le supposer.
- **Adversarial** : tester le modèle sur une image hors distribution (ni chat ni chien) forcerait quand même une prédiction avec une proba souvent élevée — aucun mécanisme "rien de connu" n'existe dans cette architecture binaire. C'est une limite à documenter, pas un bug à corriger.

**C'est la dernière phase du TP.** Le repo doit maintenant contenir : les 12 notebooks (phases 1.1 à 3.5), les courbes PNG, les fichiers `.keras`/`.tflite`, et un historique de commits qui raconte la progression.